# 03 · Simple tagging baselines

**Question:** How much can inexpensive, understandable rules achieve before we train a model?

We compare a most-common-tag rule, German keyword rules, and reuse of earlier event tags. The saved split stays unchanged. We fit only on training records and evaluate on validation records. Final-test labels are not used. The reviewed pilot is a development diagnostic: we have already used these examples to discuss the rubric, and some of the reviewer’s accepted labels differ from the written rubric.

Run every cell in order using the DemoLogic kernel. Inputs remain private; clear all outputs before committing. This notebook writes one timestamped private report per run under `reports/baselines/`.


## 1. Load the development data

The experiment is written in the cells below: load data, define rules, predict, score, inspect errors, and save a report. Run from top to bottom. You can inspect intermediate variables at every step.

We share only the snapshot/benchmark checks, tag catalog, display helpers and scoring functions. These keep input validation and metric definitions consistent across notebooks. There is no service or API involved. We will build a service only after evaluating and selecting an approach.

### Count an event type once (DR-40)

The main comparison uses **one original record per saved event group**: 732 training,
188 validation and 123 final-test representatives. The original 2,626 / 659 / 149
occurrences remain in the snapshot and locked split. We choose the earliest date,
then the lowest event ID if dates tie. In the 40 pilot groups, we keep the record
already selected for review. Neither tags nor model scores choose a record.

We use the selected record's original title and historical tags. If a group has
different title variants or historical tag sets, we retain that fact as a limitation;
we do not combine labels or replace the title with the group's display name.
There are 42 training and four validation groups with conflicting historical tags.
The private selection manifest preserves all member IDs, and every report records
its fingerprint. Earlier occurrence-based reports remain historical experiments
and must not be compared directly with these new results.


In [ ]:
import json
import platform
import re
import sys
import unicodedata
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from statistics import median
from time import perf_counter


root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Open this notebook inside the DemoLogic repository")
sys.path.insert(0, str(root / "src"))
sys.path.insert(0, str(root))

from notebooks.tools import file_sha256, notebook_sha256, percent
from notebooks.tools import show_table as show


from demo_logic.benchmark import _csv, load_benchmark, representative_selection
from demo_logic.evaluation import multilabel_metrics, parse_review_tags
from demo_logic.notebook_support import save_private_json
from demo_logic.snapshot import load_snapshot
from demo_logic.taxonomy import topic_tags

In [ ]:
rows, source = load_snapshot(root / "data/dataset.jsonl", root / "data/manifest.json")
bundle = load_benchmark(root / "data/benchmark", rows, source["content_sha256"])
if bundle.manifest.get("status") != "frozen" or source["taxonomy_version"] != "v2":
    raise ValueError("Notebook 03 requires locked split membership and taxonomy v2")
by_id = {row["external_id"]: row for row in rows}
development = {name: [] for name in ("training", "validation")}
for assignment in bundle.manifest["assignments"]:
    if assignment["split"] in development:
        development[assignment["split"]].append(
            {**by_id[assignment["external_id"]], "group_id": assignment["group_id"]}
        )
selection = representative_selection(bundle)
selection_path = save_private_json(
    root / "data/benchmark" / f"{selection['version']}.json", selection
)
selected_ids = {item["external_id"] for item in selection["records"]}
# Keep full validation history only for the separate repeat-event simulation.
validation_occurrences = development["validation"]
training, validation = (
    [row for row in development[name] if row["external_id"] in selected_ids]
    for name in ("training", "validation")
)
assert set(bundle.pilot_ids).issubset({row["external_id"] for row in validation})
tags = sorted(topic_tags(source["taxonomy_version"]))
pilot = {
    row["external_id"]: parse_review_tags(row["reviewed_tags"], set(tags))
    for row in _csv(bundle.paths["rubric_pilot"], {"external_id", "reviewed_tags"})
}

In [ ]:
show(
    ["Input", "Value"],
    [
        ("Split membership", "locked"),
        ("Training records", len(training)),
        ("Validation records", len(validation)),
        ("Reviewed development pilot", len(pilot)),
        ("Taxonomy", source["taxonomy_version"]),
    ],
)

## 2. Define the rules we want to compare

These functions belong to this notebook while we explore the baselines. They have no fitted neural model and make no external calls. The rules and examples below are visible so a reader can understand how a suggestion is produced.

### 2.1 Choose tags from the scores

Take up to two tags with positive scores, including extra tags tied exactly at second place. We also compare a strict two-tag limit, breaking ties alphabetically by slug. For regex and history, a score of 1 means a match; these scores are not probabilities. Reviewer reference labels are never shortened.

In [ ]:
BASELINE_VERSION = "simple-baselines-v3-event-types"


def select_tags(
    scores: dict[str, float], *, include_ties: bool = True, limit: int = 2
) -> list[str]:
    """Select positive heuristic scores, retaining exact cutoff ties if requested.

    Scores are counts or binary matches, never probabilities. A stable slug sort
    breaks ties for the strict-limit comparison. Empty matches mean abstention.
    """
    if limit < 1:
        raise ValueError("Tag limit must be positive")
    ranked = sorted(
        ((tag, value) for tag, value in scores.items() if value > 0), key=lambda x: (-x[1], x[0])
    )
    chosen = ranked[:limit]
    if include_ties and len(ranked) > limit:
        chosen += [item for item in ranked[limit:] if item[1] == chosen[-1][1]]
    return [tag for tag, _ in chosen]

### 2.2 Most-common tag

Count each tag once per training record, then suggest the most frequent tag for every validation event. This rule ignores the title and gives us a simple starting point. Frequency ties are resolved alphabetically.

In [ ]:
def fit_most_common(training_rows: list[dict]) -> str:
    """Choose one tag using training assignments only; break frequency ties by slug.

    Each tag counts once per event. Raises ValueError when training has no labels.
    The baseline deliberately recommends exactly one tag, even on multi-topic events.
    """
    counts = Counter(tag for row in training_rows for tag in set(row["tag_slugs"]))
    if not counts:
        raise ValueError("Most-common baseline needs labeled training records")
    return min(counts, key=lambda tag: (-counts[tag], tag))


started = perf_counter()
common = fit_most_common(training)
fit_seconds = perf_counter() - started
print("Most common training tag:", common)

### 2.3 Demo Radar's existing regex rules

This baseline uses a fixed copy of `TAG_RULES` from Demo Radar's `backend/services/tag_bulk.py`, together with its title normalization. The source revision and file fingerprint are recorded below and in the private report. The notebook runs independently; it does not import the app or need its database. Future app changes require an explicit notebook update and a new experiment version.

The dictionary below contains the actual regex patterns, which are more precise than the concept summary. For example, `\bklima` matches the beginning of a word such as `Klima` or `Klimaschutz`. `\b` means a word boundary. A country-related international-politics pattern requires both a listed country and a listed political term. A country name alone does not match that pattern. Peace-related patterns are also specific; an arbitrary mention of peace is not enough.

As in the app, matching uses only the original title: replace ß with ss, remove accents using NFKD and ASCII conversion, lowercase and collapse whitespace. There is no extra HTML-entity decoding. One or more matching patterns give the tag a score of **1**; no match gives **0**. Matching several patterns for the same tag does not increase its score. These are match indicators, not confidence probabilities.

There is deliberately no `politics` rule: generic political words do not establish German domestic scope. The tag remains in our evaluation catalog and its missed reference labels count against recall.

**What matches the app, and what differs?** The regex candidate matches and title normalization are copied exactly. The notebook then applies the capstone's tie-inclusive and strict-two policies. The app chooses up to two tags using its catalog order and may reuse recent topic-memory labels before reaching regex matching. We therefore measure the app's regex component under the benchmark policies, rather than claim to reproduce its complete autofill workflow. With binary scores, including cutoff ties retains every matched tag when more than two match.

Try the synthetic example below. Rules can match a passing mention or produce a false positive. Existing production rules may also have influenced historical labels, so agreement with historical assignments is not independent evidence of correctness. Use the reviewed pilot as a development diagnostic and reserve the final reviewed set for the final comparison.


In [ ]:
REGEX_SOURCE = {
    "repository": "demo-radar",
    "path": "backend/services/tag_bulk.py",
    "revision": "282be15d494d8c86df5013f62726c74ee6b9bda5",
    "file_sha256": "584995dd9903c514c42b80162cd6a701897a9dc4357ee6da38539850dc3ad8a2",
    "rules_sha256": "ed20d926136f3a828263e5b2876d96217a18a5f642f4e7515fda84e56bcfc859",
    "copied_on": "2026-09-25",
}

KEYWORDS = {
    "environment": (
        "\\bklima",
        "\\bumwelt",
        "\\bnatur",
        "\\bwald",
        "\\bwasser",
        "\\bartenschutz",
        "\\bbiodivers",
        "\\bkohle\\b",
        "\\btier",
        "\\btierschutz",
        "\\bvegan",
        "\\blandwirtschaft",
        "\\bernahrung",
        "\\bmuell",
        "\\bmull",
        "\\bsauberkeit",
        "\\bfossil",
        "\\berneuerbar",
        "\\bbaum",
        "\\bbaume\\b",
        "\\basphalt",
    ),
    "mobility": (
        "\\bverkehr",
        "\\bmobilitat",
        "\\bmobilitaet",
        "\\bauto",
        "\\bbahn",
        "\\boepnv",
        "\\bopnv",
        "\\bparkplatz",
        "\\bfahrrad",
        "\\bradverkehr",
        "\\bradweg",
        "\\bverkehrswende",
        "\\bcritical mass\\b",
        "\\bfussganger",
        "\\bfußganger",
        "\\bfussverkehr",
        "\\bfussgangeruberweg",
        "\\bschulweg",
        "\\bstrassenverbreiterung",
        "\\bstrasenverbreiterung",
    ),
    "housing": (
        "\\bmiete",
        "\\bwohnen",
        "\\bwohnung",
        "\\benteign",
        "\\bverdrang",
        "\\bverdraeng",
        "\\boffentlicher raum",
        "\\boeffentlicher raum",
        "\\bpark\\b",
        "\\bplatz\\b",
        "\\bstrassenraum",
        "\\bstrasenraum",
        "\\bkiez",
        "\\bgenossenschaft",
        "\\bbauen\\b",
        "\\bstadtentwicklung",
        "\\burbane mitte",
        "\\bbaufeld",
        "\\bbauzaun",
        "\\bmitbestimmung",
        "\\bfreiraum",
    ),
    "work": (
        "\\barbeit",
        "\\bstreik",
        "\\btarif",
        "\\btarifbindung",
        "\\btvod",
        "\\btvoed",
        "\\blohn",
        "\\bgewerkschaft",
        "\\bver\\\\.?di\\b",
        "\\bbeschaftigt",
        "\\bbeschaeftigt",
    ),
    "social-issues": (
        "\\bsozial",
        "\\barmut",
        "\\bburgergeld",
        "\\bbuergergeld",
        "\\bhartz",
        "\\bobdach",
        "\\btafel",
        "\\bpflege",
        "\\bklinik",
        "\\bkrankenhaus",
        "\\bgesundheit",
        "\\bgewalthilfe",
        "\\bpartnergewalt",
        "\\bgewalt gegen",
        "\\bmissstand",
        "\\bpflegeeinrichtung",
        "\\bsoziale einrichtung",
        "\\bunterkunft",
        "\\bkrise nicht auf unserem ruecken",
        "\\bkrise nicht auf unserem rucken",
    ),
    "education": (
        "\\bbildung",
        "\\bschule",
        "\\bschulweg",
        "\\bkita",
        "\\bstudent",
        "\\bhochschule",
        "\\buni\\b",
        "\\buniversitat",
        "\\buniversitaet",
        "\\bcampus",
        "\\bwissenschaft",
        "\\bforschung",
    ),
    "lgbtq": (
        "\\blsbti",
        "\\blgbt",
        "\\bqueer",
        "\\btrans\\b",
        "\\binter\\b",
        "\\bpride\\b",
        "\\bschwul",
        "\\blesbisch",
    ),
    "international-politics": (
        "\\bfriedens(?:demo|marsch|bewegung|kundgebung|politik)",
        "\\b(?:fur|fuer) (?:den )?frieden\\b",
        "\\bweltfrieden\\b",
        "\\bkrieg",
        "\\babruest",
        "\\babrust",
        "\\bantikrieg",
        "\\bwaffenliefer",
        "\\baussenpolitik",
        "\\baußenpolitik",
        "\\bnato\\b",
        "\\bwaffenstillstand",
        "(?=.*\\b(?:iran|china|israel|gaza|ukraine|aserbaidschan|russland|palastina|palaestina|libanon)\\b)(?=.*\\b(?:wahl|regierung|gefangen|hinrichtung|verfolg|krieg|solidaritat|demokratie|menschenrecht))",
    ),
    "human-rights": (
        "\\bpolitisch\\w* gefangen",
        "\\brechtsstaat",
        "\\bgrundrecht",
        "\\bmenschenrecht",
        "\\bmenschenwurde",
        "\\bmenschenwuerde",
        "\\bwurde",
        "\\bwuerde",
        "\\bfreiheit",
        "\\brespekt",
        "\\btoleranz",
        "\\bvielfalt",
        "\\bbunt\\b",
        "\\bdatenschutz",
        "\\buberwachung",
        "\\bueberwachung",
        "\\bnetzpolitik",
        "\\bdigital",
        "\\brassismus",
        "\\brassist",
        "\\bnazi",
        "\\brechtsradikal",
        "\\brechte popul",
        "\\bquerdenker",
        "\\bverschwoerung",
        "\\bverschworung",
        "\\bantisemit",
        "\\bmigration",
        "\\basyl",
        "\\bflucht",
        "\\bgeflucht",
        "\\bgefluecht",
        "\\babschieb",
        "\\bverfolg",
        "\\bdiskrimin",
        "\\borganraub",
        "\\bkinder",
        "\\bjugend",
        "\\bfamilie",
        "\\beltern",
        "\\bsenior",
        "\\brente",
        "\\bbarriere",
        "\\binklusion",
        "\\bbehindert",
        "\\bfrauen",
        "\\bfemin",
        "\\bhumanitar",
        "\\bhumanitaer",
    ),
    "culture": (
        "\\bkultur",
        "\\bkunst",
        "\\btheater",
        "\\bmusik",
        "\\bmedien",
        "\\bgedenk",
        "\\berinner",
        "\\bjahrtag",
        "\\bhistorisch",
        "\\bstolperstein",
        "\\bliebknecht",
        "\\bluxemburg",
        "\\blenin",
        "\\bsport",
        "\\bfussball",
    ),
    "religion": (
        "\\bjesus\\b",
        "\\bkirche",
        "\\breligion",
        "\\bglaube",
        "\\bglaubens",
        "\\bchrist",
        "\\bmuslim",
        "\\bjudisch",
        "\\bjuedisch",
        "\\bjude",
        "\\bjuden",
        "\\bjudentum",
        "\\bsynagog",
        "\\bgebet",
        "\\bgebete",
        "\\bgottesh",
        "\\bgotteshaus",
        "\\bdu'a",
        "\\bdua\\b",
    ),
}

COMPILED = {
    tag: [re.compile(pattern) for pattern in patterns] for tag, patterns in KEYWORDS.items()
}


def normalize_regex_title(title: str) -> str:
    """Apply Demo Radar's title-only normalization before matching its patterns."""
    text = (title or "").replace("ß", "ss").replace("ẞ", "SS")
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii").lower()
    return re.sub(r"\s+", " ", text)


def keyword_scores(title: str) -> dict[str, int]:
    """Return one binary match per tag, using Demo Radar's regex component.

    Several matching patterns still score 1. No politics rule is supplied by
    the app. Tag selection is applied separately using the benchmark policy.
    """
    text = normalize_regex_title(title)
    return {
        tag: int(any(pattern.search(text) for pattern in patterns))
        for tag, patterns in COMPILED.items()
    }

In [ ]:
example_title = "Klimaschutz und sichere Radwege"
example_scores = keyword_scores(example_title)
show(
    ["Synthetic title", "Positive scores", "Suggested tags"],
    [
        (
            example_title,
            {tag: score for tag, score in example_scores.items() if score > 0},
            select_tags(example_scores),
        ),
    ],
)

### 2.4 Earlier tags for the same event type

Reuse the tags from the latest strictly earlier date in the same group. Ignore same-day events, and abstain if records on the latest earlier day disagree. Prediction uses the group and date, with tags from historical occurrences only.

Whole event groups stay in one split. This means training history should give no suggestions for validation groups. We test useful repeat-event behavior separately in section 5, using only earlier validation dates as history.

In [ ]:
def lookup_scores(history: list[dict], event: dict) -> dict[str, int]:
    """Reuse historical tags from the latest strictly earlier day in the same group.

    Args:
        history: Labeled historical occurrences with group_id and ISO date.
        event: Target fields; its tag_slugs, if present, are never consulted.

    Same-day events are excluded because time/approval order is not reliable.
    If latest-day occurrences disagree on their tag set, abstain. Otherwise each
    inherited tag gets binary score 1. Dates and group identity are extra inputs,
    so this is a history-assisted baseline, not a title-only model.
    """
    previous = [
        row
        for row in history
        if row["group_id"] == event["group_id"] and row["date"] < event["date"]
    ]
    if not previous:
        return {}
    latest = max(row["date"] for row in previous)
    label_sets = {tuple(sorted(set(row["tag_slugs"]))) for row in previous if row["date"] == latest}
    if len(label_sets) != 1:
        return {}
    return dict.fromkeys(next(iter(label_sets)), 1)

## 3. Run the rules and compare their suggestions

The loop below runs every rule on the same validation records under both output policies. It records predictions and timing before scoring. The common-tag rule was fitted on training records above; no fitting happens here.

In [ ]:
candidates = {
    "most_common": lambda row: {common: 1},
    "keywords": lambda row: keyword_scores(row["title"]),
    "training_history": lambda row: lookup_scores(training, row),
}
runs = []
for name, predict in candidates.items():
    for include_ties in (True, False):
        predictions, durations = [], []
        for row in validation:
            started = perf_counter()
            predicted = select_tags(predict(row), include_ties=include_ties)
            durations.append(perf_counter() - started)
            predictions.append({"external_id": row["external_id"], "predicted": predicted})
        selected = [item for item in predictions if item["external_id"] in pilot]
        runs.append(
            {
                "candidate": name,
                "policy": "include_cutoff_ties" if include_ties else "strict_two",
                "historical_agreement": multilabel_metrics(
                    [row["tag_slugs"] for row in validation],
                    [item["predicted"] for item in predictions],
                    tags,
                ),
                "reviewed_pilot": multilabel_metrics(
                    [pilot[item["external_id"]] for item in selected],
                    [item["predicted"] for item in selected],
                    tags,
                ),
                "prediction_seconds": {"median": median(durations), "max": max(durations)},
                "predictions": predictions,
            }
        )

### Read the measures

**Precision:** of all suggested tags, what proportion matches the reference?    
**Recall:** of all reference tags, how many did we recover?    
**Coverage:** what proportion of demonstrations received at least one suggestion?    
Abstentions remain in recall and coverage. F1 balances precision and recall. Exact match requires the entire tag set to agree. Undefined precision means no tags were suggested.

The first table measures agreement with historical labels on the full validation split. Those labels can contain mistakes. The second uses the reviewer’s completed pilot labels on the same selected validation records. It is more directly reviewed, but small and already familiar to us. Neither table is the final test or a deployment decision.

The project targets are 80% precision, 80% recall and 95% coverage under the same output policy. These tables help identify what the next model must improve; they do not certify those targets on unseen reviewed data.


In [ ]:
for field, label in [
    ("historical_agreement", "Historical-label agreement"),
    ("reviewed_pilot", "Reviewed pilot diagnostic"),
]:
    print(label)
    show(
        ["Rule", "Policy", "N", "Precision", "Recall", "F1", "Coverage", "Exact match", ">2 tags"],
        [
            (
                run["candidate"],
                run["policy"],
                run[field]["records"],
                *[
                    percent(run[field][key])
                    for key in [
                        "micro_precision",
                        "micro_recall",
                        "micro_f1",
                        "coverage",
                        "exact_match",
                    ]
                ],
                run[field]["more_than_two"],
            )
            for run in runs
        ],
    )

In [ ]:
import matplotlib.pyplot as plt

plot_runs = [run for run in runs if run["policy"] == "include_cutoff_ties"]
fig, ax = plt.subplots(figsize=(9, 4))
for index, key in enumerate(["micro_precision", "micro_recall", "coverage"]):
    ax.bar(
        [position + index * 0.25 for position in range(len(plot_runs))],
        [run["reviewed_pilot"][key] or 0 for run in plot_runs],
        width=0.25,
        label=key.replace("micro_", ""),
    )
ax.set_xticks(
    [position + 0.25 for position in range(len(plot_runs))], [run["candidate"] for run in plot_runs]
)
ax.set_ylim(0, 1)
ax.set_ylabel("Fraction")
ax.set_title("Reviewed pilot: development diagnostics (undefined precision drawn as 0)")
ax.legend()
fig.tight_layout()
plt.show()
plt.close(fig)

## 4. See which tags and examples need work

The keyword table includes each tag's reference count, precision and recall. A rare tag's result may depend on one example. The private error table shows the first six disagreements sorted by event ID, not hand-picked successes. Compare missing and extra suggestions with the rubric; do not automatically change human labels to match the rules.

Macro measures in the JSON report average defined values over tags present in the reference and record how many tags contributed. Use micro measures for the agreed project targets; never hide low coverage behind a precision-only comparison.


In [ ]:
keyword_run = next(
    run for run in runs if run["candidate"] == "keywords" and run["policy"] == "include_cutoff_ties"
)
show(
    ["Tag", "Pilot support", "Precision", "Recall", "F1"],
    [
        (
            tag,
            values["support"],
            percent(values["precision"]),
            percent(values["recall"]),
            percent(values["f1"]),
        )
        for tag, values in sorted(keyword_run["reviewed_pilot"]["per_tag"].items())
    ],
)
titles = {row["external_id"]: row["title"] for row in validation}
errors = []
for prediction in sorted(keyword_run["predictions"], key=lambda item: item["external_id"]):
    event_id = prediction["external_id"]
    if event_id not in pilot:
        continue
    expected = set(pilot[event_id])
    actual = set(prediction["predicted"])
    if expected != actual:
        errors.append(
            (
                event_id,
                titles[event_id],
                ", ".join(sorted(expected - actual)),
                ", ".join(sorted(actual - expected)),
            )
        )
show(["Event ID", "Original title", "Missing suggestions", "Extra suggestions"], errors[:6])

## 5. Separate experiment: repeat events over time

Here we ask a different question: if an earlier occurrence already had tags, could we reuse them? We select validation events with an earlier dated occurrence in their own group. All rules are compared on that same subset. The lookup uses only dates strictly before the event; same-day records are excluded. Conflicting labels on the latest earlier day cause abstention.

This is a simulation, not a replacement for the unseen-group comparison. We do not have approval timestamps, so we cannot prove that earlier tags were actually available on the day. Group identity also supplies information that the title-only rule does not receive. Historical tag consistency may produce strong agreement without proving topic correctness. Cold starts are reported separately and excluded only from this simulation.


This supplementary simulation keeps all 659 validation occurrences. Its occurrence-weighted scores answer a repeat-history question and are not headline event-type results.


In [ ]:
# The population is chosen by date availability, not by successful prediction.
later = [
    row
    for row in validation_occurrences
    if any(
        old["group_id"] == row["group_id"] and old["date"] < row["date"]
        for old in validation_occurrences
    )
]
simulation = []
for name, predict in {
    **candidates,
    "earlier_validation_history": lambda row: lookup_scores(validation_occurrences, row),
}.items():
    for include_ties in (True, False):
        guesses, durations = [], []
        for row in later:
            started = perf_counter()
            guesses.append(select_tags(predict(row), include_ties=include_ties))
            durations.append(perf_counter() - started)
        simulation.append(
            {
                "candidate": name,
                "policy": "include_cutoff_ties" if include_ties else "strict_two",
                "prediction_seconds": {
                    "median": median(durations) if durations else None,
                    "max": max(durations) if durations else None,
                },
                "predictions": [
                    {"external_id": row["external_id"], "predicted": guess}
                    for row, guess in zip(later, guesses)
                ],
                "metrics": multilabel_metrics([row["tag_slugs"] for row in later], guesses, tags)
                if later
                else None,
            }
        )

In [ ]:
print(
    "Later occurrences:",
    len(later),
    "| Cold starts excluded:",
    len(validation_occurrences) - len(later),
)
show(
    ["Rule", "Policy", "Precision", "Recall", "Coverage"],
    [
        (
            run["candidate"],
            run["policy"],
            *[
                percent(run["metrics"][key])
                for key in ["micro_precision", "micro_recall", "coverage"]
            ],
        )
        for run in simulation
        if run["metrics"] is not None
    ],
)

## 6. Runtime, cost and private report

Runtime below covers a prediction and tag selection in this Python process. It excludes file loading, notebook rendering and future service overhead; it is not a production latency promise. These rules make no API calls, so paid API spend is $0. Local compute cost has not been measured. Fitting the common-tag rule is reported separately.

The report records input and code hashes, keyword patterns, both output policies, metrics, timings, private predictions and limitations. Every run has a distinct timestamp; the benchmark and review files are untouched.


In [ ]:
paths = [
    selection_path,
    root / "data/manifest.json",
    bundle.paths["manifest"],
    bundle.paths["rubric_pilot"],
    root / "docs/tagging-rubric.md",
]
paths += [
    root / "src/demo_logic" / name
    for name in (
        "evaluation.py",
        "benchmark.py",
        "snapshot.py",
        "taxonomy.py",
        "notebook_support.py",
    )
]
report = {
    "baseline_version": BASELINE_VERSION,
    "selection_version": selection["version"],
    "selection_sha256": selection["selection_sha256"],
    "taxonomy_version": source["taxonomy_version"],
    "snapshot_sha256": source["content_sha256"],
    "fingerprints": {str(path.relative_to(root)): file_sha256(path) for path in paths},
    "environment": {"python": platform.python_version(), "platform": platform.platform()},
    "training_records": len(training),
    "validation_records": len(validation),
    "pilot_records": len(pilot),
    "most_common_tag": common,
    "fit_seconds": fit_seconds,
    "keyword_patterns": KEYWORDS,
    "regex_source": REGEX_SOURCE,
    "api_cost_usd": 0,
    "compute_cost_measured": False,
    "runs": runs,
    "history_simulation": {
        "records": len(later),
        "cold_start_records_excluded": len(validation_occurrences) - len(later),
        "runs": simulation,
    },
    "limitations": [
        "Historical agreement is not independently reviewed accuracy.",
        "Pilot examples informed the rubric; they are development diagnostics, not a blind test. Regex rules come from the recorded Demo Radar source.",
        "the reviewer retained some rubric disagreements in the pilot labels; those decisions are unchanged.",
        "History simulation assumes earlier labels were available; approval timestamps are absent.",
        "Prediction timings exclude loading, notebook rendering, network and service overhead.",
        "Regex scores are binary matches, not confidence probabilities. Benchmark output policies differ from app autofill selection.",
    ],
}

notebook_path = root / "notebooks/03_simple_baselines.ipynb"
report["notebook_source_sha256"] = notebook_sha256(notebook_path, scheme="cells")

In [ ]:
show(
    ["Rule", "Policy", "Median ms", "Maximum ms"],
    [
        (
            run["candidate"],
            run["policy"],
            round(run["prediction_seconds"]["median"] * 1000, 4),
            round(run["prediction_seconds"]["max"] * 1000, 4),
        )
        for run in report["runs"]
    ],
)
print("Common-tag fit seconds:", report["fit_seconds"], "| API cost: $0; compute cost not measured")
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
report["run_id"] = run_id
report_path = save_private_json(root / "reports/baselines" / f"{run_id}.json", report)
print("Private report:", report_path.relative_to(root))

## Review and next step

1. Check that the first table uses the locked split and the expected training, validation and pilot counts.
2. Compare the two output policies. Notice whether extra tied tags help recall at the expense of precision.
3. Read the six pilot disagreements against the rubric. Keep accepted labels unchanged; record concerns for discussion.
4. Confirm that training-history coverage is zero and that the repeat-event simulation is clearly separate.

These results become the baseline for the TF-IDF notebook. Continue exploring and evaluating candidates in notebooks. After the comparison identifies an approach worth using, extract its preprocessing and prediction logic into a service if time permits. Stronger models must earn their added complexity. Keep the final-test queue out of model tuning, and clear outputs before committing this notebook.
